- Johannes Glasø Lyngaas
- 549900
- [174] - Far

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import warnings
import joblib
import numpy as np

warnings.simplefilter(action='ignore', category=FutureWarning)

#### Replace the cell below with local data files

In [ ]:
# Load the AIS Training Data
bucket = 'mlprojectbucket1kjor' #remove
ais_train_path = f'gs://{bucket}/ais_train.csv' #remove
ais_test_path = f'gs://{bucket}/ais_test.csv' #remove

#Replace with this or similar
# ais_train_path = 'ais_train.csv'
# ais_test_path = 'ais_test.csv'

print("Loading training data...")
ais_data = pd.read_csv(ais_train_path, delimiter='|')
print("Training data loaded successfully. Shape:", ais_data.shape)

In [ ]:
print("Creating temporal features...")
ais_data['time'] = pd.to_datetime(ais_data['time'])
ais_data['hour_of_day'] = ais_data['time'].dt.hour
ais_data['day_of_week'] = ais_data['time'].dt.dayofweek

print("Creating lag features for vessel movement...")
ais_data['latitude_lag1'] = ais_data.groupby('vesselId')['latitude'].shift(1)
ais_data['longitude_lag1'] = ais_data.groupby('vesselId')['longitude'].shift(1)
ais_data[['latitude_lag1', 'longitude_lag1']] = ais_data[['latitude_lag1', 'longitude_lag1']].fillna(method='ffill').fillna(ais_data[['latitude', 'longitude']])
print("Lag features created.")

# Fill NaNs
ais_data[['latitude_lag1', 'longitude_lag1']] = ais_data[['latitude_lag1', 'longitude_lag1']].fillna(method='ffill').fillna(method='bfill').fillna(ais_data[['latitude', 'longitude']])

# Selecting Features and Targets
features = ['sog', 'cog', 'rot', 'hour_of_day', 'day_of_week', 'latitude_lag1', 'longitude_lag1']
y_lat = ais_data['latitude']
y_lon = ais_data['longitude']

print("Splitting data into training and validation sets...")
X = ais_data[features]
X_train, X_val, y_lat_train, y_lat_val, y_lon_train, y_lon_val = train_test_split(X, y_lat, y_lon, test_size=0.2, random_state=42)
print("Data split complete. Training set size:", X_train.shape[0], "Validation set size:", X_val.shape[0])

print("Checking training set for NaNs...")
print(X_train.isna().sum())
print("Checking validation set for NaNs...")
print(X_val.isna().sum())
print(ais_data[['latitude_lag1', 'longitude_lag1']].isna().sum())

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
print("Training Random Forest models...")
rf_lat_model2 = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_lon_model2 = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs =-1)

rf_lat_model2.fit(X_train, y_lat_train)
rf_lon_model2.fit(X_train, y_lon_train)
print("Random Forest models trained successfully.")

print("Evaluating models on validation set...")
y_lat_pred = rf_lat_model2.predict(X_val)
y_lon_pred = rf_lon_model2.predict(X_val)

lat_mse = mean_squared_error(y_lat_val, y_lat_pred)
lat_mae = mean_absolute_error(y_lat_val, y_lat_pred)
lat_r2 = r2_score(y_lat_val, y_lat_pred)

lon_mse = mean_squared_error(y_lon_val, y_lon_pred)
lon_mae = mean_absolute_error(y_lon_val, y_lon_pred)
lon_r2 = r2_score(y_lon_val, y_lon_pred)

print(f"Latitude Model - MSE: {lat_mse:.4f}, MAE: {lat_mae:.4f}, R^2: {lat_r2:.4f}")
print(f"Longitude Model - MSE: {lon_mse:.4f}, MAE: {lon_mae:.4f}, R^2: {lon_r2:.4f}")

In [ ]:
print("Loading test data...")
ais_test_data = pd.read_csv(ais_test_path, delimiter=',')
ais_test_data['time'] = pd.to_datetime(ais_test_data['time'])
ais_test_data['hour_of_day'] = ais_test_data['time'].dt.hour
ais_test_data['day_of_week'] = ais_test_data['time'].dt.dayofweek

In [ ]:
print("Merging test data with latest known positions...")
latest_positions = ais_data.groupby('vesselId')[['latitude', 'longitude']].last().reset_index()
ais_test_data = pd.merge(ais_test_data, latest_positions, on='vesselId', how='left')
print("Test data merged with latest positions. Shape:", ais_test_data.shape)

print("Preparing test data for prediction...")
X_test_final = pd.DataFrame({
    'sog': [5] * len(ais_test_data),  # Assume default speed over ground of 5 knots
    'cog': [180] * len(ais_test_data), # Assume default course over ground of 180 degrees
    'rot': [0] * len(ais_test_data),   # Assume no rate of turn (straight movement)
    'hour_of_day': ais_test_data['hour_of_day'],
    'day_of_week': ais_test_data['day_of_week'],
    'latitude_lag1': ais_test_data['latitude'],
    'longitude_lag1': ais_test_data['longitude']
})
print("Test data prepared for prediction. Shape:", X_test_final.shape)

print("Making predictions for test data...")
latitude_predicted = rf_lat_model2.predict(X_test_final)
longitude_predicted = rf_lon_model2.predict(X_test_final)
print("Predictions complete.")

print("Creating submission file...")
submission = pd.DataFrame({
    'ID': ais_test_data['ID'],
    'longitude_predicted': longitude_predicted,
    'latitude_predicted': latitude_predicted
})

# Save Submission to CSV
submission_path = 'RFcopy1.csv'
submission.to_csv(submission_path, index=False)
print(f"Submission file created at {submission_path}")